# Chapter 18: Decomposition and Multi-Hop Retrieval

Two-hop obligations, join identity/source support; incomplete or wrong bridges rejected.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from document_identity import Collection, digest_bytes
from evidence_units import evidence_success, version_key
from query_retrieval import Retrieval, interpret, reformulate, Budget
import json
from pathlib import Path

fixture = json.loads((ROOT / "experiments" / "part4-complex" / "development-corpus.json").read_text())
relations = json.loads((ROOT / "experiments" / "part4-complex" / "development-relations.json").read_text())
config = json.loads((ROOT / "experiments" / "part4-complex" / "config.json").read_text())
needs = json.loads((ROOT / "experiments" / "part4-complex" / "development-queries.json").read_text())["queries"]

def engine(view="main", cfg=None):
    return Retrieval(Collection(fixture), view, relations, cfg if cfg is not None else config)


In [ ]:
# A complete supported two-hop path is accepted with exact edge provenance.
e = engine()
p = e.plan("Which contact serves project PR-21?", "q")
out = e.execute(p, "structured")
print("path:", [x["edge_id"] for x in out["path"]])
print("status:", out["status"], "calls:", out["cost"]["calls"])
assert [x["edge_id"] for x in out["path"]] == ["development:assignment", "development:contact"]
assert all(x["support"]["quote"].startswith("REL ") for x in out["path"])
assert out["cost"]["calls"] == 2


In [ ]:
# Search hops require retrieving the supporting units, not just declaring edges.
e = engine()
p = e.plan("Which contact serves project PR-21?", "q")
out = e.execute(p, "multi-hop")
print("path:", [x["edge_id"] for x in out["path"]], "calls:", out["cost"]["calls"])
assert len(out["path"]) == 2
assert out["cost"]["calls"] == 2


In [ ]:
# Incomplete chains are rejected: missing first or second hop yields no path.
for view in ("missing-first", "missing-second"):
    e = engine(view)
    p = e.plan("Which contact serves project PR-21?", "q")
    out = e.execute(p, "structured")
    print(view, "-> path:", out["path"], "status:", out["status"])
    assert out["path"] == []


In [ ]:
# Wrong or borrowed bridges are rejected.
e = engine()
# PR-28 has a wrong bridge; PR-29 only a mention edge.
for query in ("Which contact serves project PR-28?", "Which contact serves project PR-29?"):
    p = e.plan(query, "q")
    out = e.execute(p, "structured")
    print(repr(query), "->", out["path"], out["status"])
    assert out["path"] == []


In [ ]:
# Conflicting supported relations block acceptance even when positive statements exist.
e = engine("conflicted")
p = e.plan("Which contact serves project PR-21?", "q")
out = e.execute(p, "structured")
print("status:", out["status"], "path:", out["path"])
assert out["path"] == []
assert out["status"] == "conflicting_supported_relations"


In [ ]:
# Depth and cycle controls bound traversal.
e = engine()
p = e.plan("PR-21", "q")
print("depth-bound result:", e.navigate(p, Budget(p["budgets"]), predicates=("operated_by", "contact", "contact"))[1])
assert e.navigate(p, Budget(p["budgets"]), predicates=("operated_by", "contact", "contact"))[1] == []
# cycle view
config2 = json.loads(json.dumps(config))
config2["budgets"].update(graph_depth=3, max_calls=3, candidate_rows=9)
ec = engine("cycle", cfg=config2)
pc = ec.plan("PR-21", "q")
print("cycle status:", ec.navigate(pc, Budget(pc["budgets"]), predicates=("operated_by", "contact", "contact"))[2])
assert ec.navigate(pc, Budget(pc["budgets"]), predicates=("operated_by", "contact", "contact"))[2] == "cycle"


In [ ]:
# Public projection rechecks edges and drops restricted/conflicted paths.
e = engine()
p = e.plan("Which contact serves project PR-21?", "q")
out = e.execute(p, "structured")
public = e.public(out, "restricted")
print("public path:", public["path"])
assert public["path"] == []


## Observed versus not established
Observed: supported two-hop paths, exact provenance, rejection of incomplete/wrong/conflicting/cyclic bridges, and depth bounds. Not established: general relation extraction, transitive inference, or answer correctness.